# Vocal Tract Demo — OLA FIR (`pink_trombone_ola`)

Same examples as `tract_demo.ipynb`, but uses the OLA FIR approximation instead of
the sequential waveguide.  Each cell compares both methods:

- **Reference** (`pink_trombone`): exact sequential Kelly-Lochbaum waveguide
- **OLA FIR** (`pink_trombone_ola`): one impulse response per control frame, overlap-add convolution

See the bottom of this notebook for a speed benchmark.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time

import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Audio, display

from samuel.pink_trombone import (
    pink_trombone,
    pink_trombone_ola,
    PARAM_NAMES,
    N_PARAMS,
    SAMPLE_RATE,
    SAMPLES_PER_FRAME,
    _compute_diameter_profile,
    _TRACT_N,
)

In [ ]:
CONTROL_RATE = SAMPLE_RATE / SAMPLES_PER_FRAME


def make_params(duration_s=2.0, **overrides):
    T = int(duration_s * CONTROL_RATE)
    defaults = {
        "frequency": 350,
        "voiceness": 0.6,
        "intensity": 1,
        "tongueIndex": 12.9,
        "tongueDiameter": 2.43,
        "vibratoWobble": 0,
        "vibratoFrequency": 6,
        "vibratoGain": 0.0,
        "tractLength": 44,
        "constrictionIndex": 44.0,
        "constrictionDiameter": 3.0,
    }
    params = torch.zeros(1, T, N_PARAMS)
    for i, name in enumerate(PARAM_NAMES):
        val = overrides.get(name, defaults.get(name, 0))
        if isinstance(val, torch.Tensor):
            params[0, :, i] = val[:T]
        else:
            params[0, :, i] = val
    return params


def play_both(params, seed=0, ir_length=4096, normalize=True, label=None):
    """Synthesize with both methods, print timing, and play both."""
    with torch.no_grad():
        t0 = time.perf_counter()
        ref = pink_trombone(params, seed=seed)
        t_ref = time.perf_counter() - t0

        t0 = time.perf_counter()
        ola = pink_trombone_ola(params, seed=seed, ir_length=ir_length)
        t_ola = time.perf_counter() - t0

    if label:
        print(f"── {label} ──")

    ref_rms = ref.pow(2).mean().sqrt().item()
    err_rms = (ref - ola).pow(2).mean().sqrt().item()
    rel_err = err_rms / (ref_rms + 1e-9)
    print(f"  reference: {t_ref*1000:.0f} ms  |  OLA FIR: {t_ola*1000:.0f} ms  |  relative error: {rel_err:.3f}")

    def norm(w):
        return w / w.abs().max().clamp(min=1e-6) * 0.9 if normalize else w

    print("  Reference:")
    display(Audio(norm(ref[0]).numpy(), rate=SAMPLE_RATE))
    print("  OLA FIR:")
    display(Audio(norm(ola[0]).numpy(), rate=SAMPLE_RATE))
    return ref, ola

## 1. Vowel sounds

Different tongue positions produce different vowels.

In [ ]:
vowels = {
    "/ɑ/ (father)": dict(tongueIndex=12.9, tongueDiameter=2.43),
    "/i/ (feet)": dict(tongueIndex=27.2, tongueDiameter=2.05),
    "/u/ (boot)": dict(tongueIndex=22.8, tongueDiameter=2.05),
    "/ɛ/ (bed)": dict(tongueIndex=20.0, tongueDiameter=2.3),
    "/ʌ/ (but)": dict(tongueIndex=17.0, tongueDiameter=2.3),
}

for label, kw in vowels.items():
    params = make_params(duration_s=1.5, frequency=350, voiceness=0.6, **kw)
    play_both(params, label=label)

## 2. Waveform comparison

Visual comparison of reference vs OLA FIR on a single vowel.

In [ ]:
params = make_params(duration_s=1.0, frequency=200, voiceness=0.6,
                     tongueIndex=27.2, tongueDiameter=2.05,
                     vibratoGain=0.0, vibratoWobble=0.0)
ref, ola = play_both(params, seed=0, label="/i/ waveform comparison")

# Show a short window
start, end = 2 * SAMPLES_PER_FRAME, 2 * SAMPLES_PER_FRAME + 800
t = (torch.arange(end - start) / SAMPLE_RATE * 1000).numpy()  # ms

err = (ola[0, start:end] - ref[0, start:end]).numpy()

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05)
fig.add_trace(go.Scatter(x=t, y=ref[0, start:end].numpy(), name="reference",
                         line=dict(width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=ola[0, start:end].numpy(), name="OLA FIR",
                         line=dict(width=1, color="#ff7f0e")), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=err, name="error",
                         line=dict(width=1, color="#d62728")), row=3, col=1)
fig.update_yaxes(title_text="reference", row=1, col=1)
fig.update_yaxes(title_text="OLA FIR", row=2, col=1)
fig.update_yaxes(title_text="error", row=3, col=1)
fig.update_xaxes(title_text="time (ms)", row=3, col=1)
fig.update_layout(height=500, width=1000,
                  title_text="/i/ — reference vs OLA FIR (static params, 18 ms window)")
fig.show()

print(f"peak |error|: {abs(err).max():.4f}  |  signal peak: {ref[0, start:end].abs().max():.4f}")

## 3. Spectrogram comparison

In [ ]:
import numpy as np
from scipy.signal import spectrogram

params = make_params(duration_s=1.0, frequency=200, voiceness=0.6,
                     tongueIndex=27.2, tongueDiameter=2.05,
                     vibratoGain=0.0, vibratoWobble=0.0)

with torch.no_grad():
    ref = pink_trombone(params, seed=0)[0].numpy()
    ola = pink_trombone_ola(params, seed=0, ir_length=4096)[0].numpy()

fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=("Reference", "OLA FIR"),
                    horizontal_spacing=0.05)

for col, sig in [(1, ref), (2, ola)]:
    f, t_spec, Sxx = spectrogram(sig, fs=SAMPLE_RATE, nperseg=1024, noverlap=512)
    Sxx_db = 10 * np.log10(Sxx + 1e-12)
    fig.add_trace(
        go.Heatmap(z=Sxx_db, x=t_spec, y=f, colorscale="Inferno",
                   showscale=(col == 2), colorbar=dict(title="dB")),
        row=1, col=col,
    )

fig.update_yaxes(range=[0, 5000], title_text="frequency (Hz)", row=1, col=1)
fig.update_yaxes(range=[0, 5000], row=1, col=2)
fig.update_xaxes(title_text="time (s)", row=1, col=1)
fig.update_xaxes(title_text="time (s)", row=1, col=2)
fig.update_layout(height=400, width=1100)
fig.show()

## 4. Vowel glide (time-varying params)

Sweep tongue index smoothly /i/ → /ɑ/.  The OLA approximation uses per-frame
filters sampled at frame midpoints, so slowly-varying params should still be close.

In [ ]:
T = 38  # ~3 seconds
params = make_params(
    duration_s=T / CONTROL_RATE,
    tongueIndex=torch.linspace(27.2, 12.9, T),
    tongueDiameter=torch.linspace(2.05, 2.43, T),
    frequency=350,
    voiceness=0.6,
)
play_both(params, label="Glide /i/ → /ɑ/")

## 5. Constriction sweep

Open → closed: from vowel to stop consonant.

In [ ]:
T = 38
params = make_params(
    duration_s=T / CONTROL_RATE,
    frequency=140,
    voiceness=0.6,
    tongueIndex=12.9,
    tongueDiameter=2.43,
    constrictionIndex=30.0,
    constrictionDiameter=torch.linspace(2.0, -0.5, T),
)
play_both(params, label="Constriction sweep: open → closed")

## 6. Fricative

Turbulence path: narrow constriction mid-tract.

In [ ]:
params = make_params(
    duration_s=0.5,
    frequency=140,
    voiceness=0.1,
    constrictionIndex=32.0,
    constrictionDiameter=0.5,
)
play_both(params, label="Fricative mid-tract (c_idx=32, c_diam=0.5)")

## 7. Speed benchmark

Compare wall-clock time across clip lengths.

In [ ]:
import gc

durations = [1, 2, 4, 8]
results = []

for dur in durations:
    params = make_params(duration_s=dur, frequency=350, voiceness=0.6,
                         tongueIndex=27.2, tongueDiameter=2.05,
                         vibratoGain=0.0, vibratoWobble=0.0)
    times = {}
    for name, fn in [("reference", pink_trombone), ("OLA FIR", pink_trombone_ola)]:
        # warm-up
        with torch.no_grad():
            fn(params, seed=0)
        gc.collect()
        t0 = time.perf_counter()
        with torch.no_grad():
            for _ in range(3):
                fn(params, seed=0)
        times[name] = (time.perf_counter() - t0) / 3

    results.append((dur, times))
    print(f"{dur}s clip: reference={times['reference']*1000:.0f} ms, "
          f"OLA FIR={times['OLA FIR']*1000:.0f} ms, "
          f"speedup={times['reference']/times['OLA FIR']:.1f}x")

fig = go.Figure()
fig.add_trace(go.Scatter(x=durations, y=[r[1]["reference"]*1000 for r in results],
                         mode="lines+markers", name="reference",
                         marker=dict(symbol="circle", size=8)))
fig.add_trace(go.Scatter(x=durations, y=[r[1]["OLA FIR"]*1000 for r in results],
                         mode="lines+markers", name="OLA FIR",
                         marker=dict(symbol="square", size=8)))
fig.update_layout(
    title="Synthesis time vs clip length (CPU)",
    xaxis_title="clip duration (s)",
    yaxis_title="wall time (ms)",
    width=700, height=400,
)
fig.show()

## 8. Accuracy vs IR length

Shorter IRs are faster but less accurate.  The 0.999-per-sample damping means the
impulse response decays to ~1.7% of its initial amplitude at 4096 taps (~93 ms).

In [ ]:
params = make_params(duration_s=0.4, frequency=200, voiceness=0.6,
                     tongueIndex=27.2, tongueDiameter=2.05,
                     vibratoGain=0.0, vibratoWobble=0.0)

with torch.no_grad():
    ref = pink_trombone(params, seed=0)

# Dense log-spaced sweep; audio is only played for a small representative subset.
ir_lengths = [32, 48, 64, 96, 128, 192, 256, 384, 512, 768,
              1024, 1536, 2048, 3072, 4096, 6144, 8192]
audio_lengths = {128, 512, 2048, 8192}
errors = []

skip = SAMPLES_PER_FRAME
ref_seg = ref[:, skip:]
# Spectral relative error: compare magnitude spectra, which ignores phase
# misalignment between the exact waveguide and the per-frame OLA approximation.
ref_mag = torch.fft.rfft(ref_seg, dim=-1).abs()
ref_spec_norm = ref_mag.pow(2).mean().sqrt().item()

for L in ir_lengths:
    with torch.no_grad():
        ola = pink_trombone_ola(params, seed=0, ir_length=L)
    ola_mag = torch.fft.rfft(ola[:, skip:], dim=-1).abs()
    err_spec = (ola_mag - ref_mag).pow(2).mean().sqrt().item()
    rel = err_spec / ref_spec_norm
    errors.append(rel)
    print(f"L={L:5d}: spectral relative error = {rel:.4f} ({rel*100:.2f}%)")
    if L in audio_lengths:
        display(Audio(ola[0].numpy(), rate=SAMPLE_RATE))

fig = go.Figure()
fig.add_trace(go.Scatter(x=ir_lengths, y=[e * 100 for e in errors],
                         mode="lines+markers", marker=dict(size=8)))
fig.update_layout(
    title="OLA FIR accuracy vs IR length (spectral magnitude)",
    xaxis=dict(title="IR length (taps)", type="log", showgrid=True),
    yaxis=dict(title="spectral relative error (%)", type="log", showgrid=True),
    width=700, height=420,
)
fig.show()

## 9. Gradient comparison

Gradients of silence-loss w.r.t. each parameter, for both methods.

In [ ]:
params_ref = make_params(duration_s=2.0)
params_ola = params_ref.clone()
params_ref.requires_grad_(True)
params_ola.requires_grad_(True)

loss_ref = pink_trombone(params_ref, seed=0).pow(2).mean()
loss_ref.backward()

loss_ola = pink_trombone_ola(params_ola, seed=0).pow(2).mean()
loss_ola.backward()

grad_ref = params_ref.grad[0].abs().mean(dim=0).detach()
grad_ola = params_ola.grad[0].abs().mean(dim=0).detach()

fig = go.Figure()
fig.add_trace(go.Bar(x=PARAM_NAMES, y=grad_ref.numpy(), name="reference"))
fig.add_trace(go.Bar(x=PARAM_NAMES, y=grad_ola.numpy(), name="OLA FIR",
                     opacity=0.8))
fig.update_layout(
    barmode="group",
    title="Parameter gradients (silence loss, 2 s clip)",
    yaxis_title="|∂loss/∂param|",
    xaxis=dict(tickangle=-45),
    width=1000, height=450,
)
fig.show()